# 🧠 Gemma 4 — Direct weight-space policy experiment (isolated)

**Research branch only. Not part of Merlino's governed workshop or Experiment 8.**

**Question:** Can numerical edits to Gemma 4 E2B's weights encode a *new* fictional weekly-increase policy (10% → 5%), with no gradient descent, when the policy is absent or stale in the prompt?

**Truth boundary:** The knowledge graph and SHACL remain authoritative. Gate = **validation only**. An LLM/human may author a patch; the patched model may propose; the existing Merlino gate accepts or rejects. **No inference success is claimed in this notebook.**

We compare base, prompt-only, patched, stale-prompt and randomly perturbed controls. All data refer to fictional runner Alex.

[Source: Weight-Space Programming, Takehi et al. (2026)](https://rikiyat.github.io/blog/weight-space-programming/) — their Qwen finding is *not* a Gemma policy-edit result.

## 0 · Research environment

**GPU:** Choose *A100 (40 GB)* if available, not a T4 by default. Gemma 4 E2B at BF16 is about 11.4 GB of model inference memory, but the full workbench needs headroom.

Because the GitHub repository is **private**, download a **ZIP of the research branch** from GitHub and upload it below. This does **not** upload personal running data. Colab is Google-hosted compute, not private/on-device execution. Do not paste GitHub/Hugging Face credentials into notebook cells. If weights require license acceptance, complete that step in your Hugging Face account first.

In [ ]:
import os, sys, zipfile, tempfile
from pathlib import Path
from google.colab import files
print('Upload the ZIP of the PRIVATE research/gemma-weight-space-policy-v0 GitHub branch.')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one repository ZIP')
zip_name = next(iter(uploaded))
if not zip_name.endswith('.zip') or len(uploaded[zip_name]) > 50_000_000:
    raise ValueError('Expected a repository ZIP smaller than 50 MB')
workspace = Path('/content/weight-space-policy').resolve()
workspace.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as archive:
    for member in archive.infolist():
        dest = (workspace / member.filename).resolve()
        if not dest.is_relative_to(workspace) or member.file_size > 15_000_000:
            raise ValueError('Unsafe ZIP entry')
    archive.extractall(workspace)
candidates = list(workspace.rglob('experiments/weight_space_policy/run.py'))
if len(candidates) != 1:
    raise RuntimeError(f'Expected one experimental module, found {len(candidates)}')
PROJECT_ROOT = candidates[0].parents[2]
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('Project:', PROJECT_ROOT)


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install',
                       '-r', 'experiments/weight_space_policy/requirements-research.txt'])
subprocess.run([sys.executable, '-m', 'pytest', '-q',
                'experiments/weight_space_policy/tests'], check=True)


## 1 · Establish the policy oracle (CPU + real SHACL)

The current **synthetic** policy is 5%, which caps the 33 km history at **34.65 km**. A proposal of 34.6 km can pass; 34.7 and 42.5 must be blocked. This check is **independent of any Gemma weights**. No week can be started or approved by this experiment.

In [ ]:
from experiments.weight_space_policy.run import shacl_check
from pprint import pprint
for proposed in (34.6, 34.7, 42.5):
    result = shacl_check(proposed)
    print(f'{proposed:.1f} km: accepted={result["accepted"]}')
    if proposed == 34.6: assert result['accepted']
    else: assert not result['accepted']


## 2 · Inspect actual weights before proposing an edit

Unlike prompt engineering, a **weight-space** programmer must choose which tensor values to modify. Inspect named, eligible dense MLP projections and, optionally, **bounded raw weight windows**. These tools intentionally do not load/save modified checkpoints, perform training, or evaluate a model under the hood.

**No candidate patch is supplied or claimed to work.** An editing LLM/human should examine the diagnostic data and failed synthetic prompts, write down a mechanism hypothesis, and propose a patch JSON. This is the research intervention being tested.

In [ ]:
from experiments.weight_space_policy.run import load_gemma
from experiments.weight_space_policy.patch import list_editable_weights
from experiments.weight_space_policy.inspect import weight_window
MODEL_ID = 'google/gemma-4-E2B-it'
from huggingface_hub import HfApi
MODEL_REVISION = HfApi().model_info(MODEL_ID).sha  # Resolve and PIN the exact checkpoint revision
print('Pinned Gemma checkpoint:', MODEL_REVISION)
USE_GPU = False  # Change to True ONLY on a GPU-enabled runtime after accepting model terms
model = processor = None
if USE_GPU:
    model, processor = load_gemma(MODEL_ID, revision=MODEL_REVISION)
    candidates = list_editable_weights(model, limit=24)
    for c in candidates[:12]: print(c)
    if not candidates:
        raise RuntimeError('No editable dense MLP projections found')
    # Read a genuine 4x4 numerical window; inspect names and shapes before editing.
    pprint(weight_window(model, parameter=candidates[0]['parameter'], row=0, col=0, height=4, width=4))
else:
    print('CPU-only review mode. GPU weight reading not executed.')


### Baseline first: diagnose Gemma before modifying any weights

Run the **unmodified checkpoint** under no-policy, stale-policy and current-policy prompts. Save all raw responses and SHACL verdicts. This establishes the failure pattern used to select a weight-edit hypothesis. These examples are development data, **not** held-out evidence of generalization.

The gate only evaluates proposals; it does not repair or select any weight edit. GPU only, synthetic data only.

In [ ]:
from experiments.weight_space_policy.run import PROBES, prompt_for, generate, extract_proposal, shacl_check
import json
BASE_DIAGNOSTICS = []
if USE_GPU:
    for policy_context in ('none', 'old', 'current'):
        for probe in PROBES:
            raw = generate(model, processor, prompt_for(probe, policy_context))
            km, parse_status = extract_proposal(raw)
            verdict = shacl_check(km)
            row = {'probe': probe['id'], 'prompt_policy': policy_context,
                   'requested_km': probe['request'], 'proposed_km': km,
                   'parse_status': parse_status, 'gate_pass': verdict['accepted'],
                   'violations': verdict['violations'], 'raw_output': raw}
            BASE_DIAGNOSTICS.append(row)
            print(policy_context, probe['id'], 'requested:', probe['request'],
                  'proposed:', km, 'gate_pass:', verdict['accepted'])
    diagnostic_file = PROJECT_ROOT / 'artifacts' / 'wsp-base-diagnostics.json'
    diagnostic_file.parent.mkdir(exist_ok=True)
    diagnostic_file.write_text(json.dumps({'model_id': MODEL_ID,
        'model_revision': MODEL_REVISION, 'development_data': True,
        'records': BASE_DIAGNOSTICS}, indent=2), encoding='utf-8')
    print('Saved development diagnostics:', diagnostic_file)
else:
    print('Enable USE_GPU in the previous cell before collecting baseline diagnostics.')


## 3 · Author and audit the numerical patch

After inspection, create `candidate_patch.json` on your machine or via a separate weight-programming assistant. It must include a **natural-language hypothesis**, actual tensor names and bounded numeric edits.

Allowed: scale selected MLP projection rows/columns (0.8–1.2), or apply explicit `[[row, col, delta], ...]` additions (up to 4,096 values per tensor), all with automatic rollback. Forbidden: arbitrary Python execution, hidden optimization/training, changing SHACL or the KG, modifying quantized weights.

**Do not claim a demonstrated Gemma policy edit until the GPU trial produces evidence.**

In [ ]:
from google.colab import files
import json
from experiments.weight_space_policy.patch import parse_manifest, canonical_digest
PATCH_FILE = None
if USE_GPU:
    uploaded_patch = files.upload()   # choose candidate_patch.json
    if len(uploaded_patch) != 1:
        raise ValueError('Upload one human-reviewed JSON manifest')
    patch_name = next(iter(uploaded_patch))
    PATCH_FILE = PROJECT_ROOT / patch_name
    candidate = json.loads(PATCH_FILE.read_text(encoding='utf-8'))
    parse_manifest(candidate)
    print('Patch SHA256:', canonical_digest(candidate))
else:
    print('No intervention was authored. No GPU results can be claimed.')


## 4 · Six-arm trial, independent SHACL on every proposal

Five smoke probes × six conditions = **30 model responses**. The edited model is evaluated against a base/no-policy model, stale 10% prompt, current 5% prompt, and random-index-edit control. A patch affects only model weights; it never changes ontology, policy, gate or authorization. `temporary_patch` **restores the original parameters after each arm**.

Passing the gate is not proof of policy being stored in weights: a uniformly conservative model might pass without knowing the exact 5% threshold.

In [ ]:
from experiments.weight_space_policy.run import run_eval
import torch, gc
RESULT_FILE = PROJECT_ROOT / 'artifacts' / 'wsp-policy-smoke.json'
if USE_GPU and PATCH_FILE is not None:
    # Release inspection copy before the evaluator loads its own same-checkpoint copy.
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    results = run_eval(
        patch_manifest=json.loads(PATCH_FILE.read_text(encoding='utf-8')),
        model_id=MODEL_ID, revision=MODEL_REVISION, out_path=RESULT_FILE)
    pprint(results['summary'])
    print('Results:', RESULT_FILE)
else:
    print('GPU inference intentionally not performed. Review methods and CPU tests above.')


## 5 · How to interpret results

A meaningful gain requires edited/no-policy proposals to pass **more often than both** base/no-policy and randomized-edit/no-policy, while preserving unrelated capabilities. The base/current-policy arm establishes how well an explicit prompt works. The edited/old-policy arm tests resistance to stale instructions.

This smoke test is **not sufficient** to establish a reliable weight-encoded policy. The next stage requires 30+ held-out prompts, new graph-derived baseline magnitudes, multiple independent edit candidates, norm-matched controls, and unrelated-task regressions.

In all outcomes, **the SHACL gate validates; it never repairs**. Policy authority and human approval remain outside the model's weights.

Only after meaningful, replicated improvements should we consider connecting the weight programmer to Hippocampus research or memory-navigation traces.

In [ ]:
if USE_GPU and PATCH_FILE is not None:
    from google.colab import files
    files.download(str(RESULT_FILE))
else:
    print('No result file produced in CPU-only review mode.')
